In [1]:
import datetime, calendar, glob, os, requests, pickle

import pickle as pkl
import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.colors as mcolors
import networkx as nx

from matplotlib import cm
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.transforms import ScaledTranslation

from datetime import datetime

from shapely.geometry import box
from shapely.ops import unary_union

plt.rcParams["legend.handlelength"] = 1
plt.rcParams["legend.handleheight"] = 1.125
plt.rcParams["font.family"] = "Avenir"

path_to_data = "/Users/Guille/Desktop/dynamic_update/data"
path_to_images = "/Users/Guille/Desktop/dynamic_update/images"
path_to_params = "/Users/Guille/Desktop/dynamic_update/params"
path_to_validation = "/Users/Guille/Desktop/dynamic_update/validation"

# Loading color palette
palette_ = pd.read_csv(path_to_data + "/palette.csv")
print(palette_)

# Loading Texas map
TX_ = gpd.read_file(path_to_data + "/maps/TX/State.shp")

T = 288
resource = 'wind'

      miro      ibm
0  #013396  #648FFF
1  #B1C06E  #785EF0
2  #056534  #DC267F
3  #F80202  #FE6100
4  #FDD906  #FFB000
5  #FCF795      NaN
6  #CCEDFF      NaN
7  #FDD60B      NaN
8  #FCE9D0      NaN


In [2]:
# Load
TX_ = gpd.read_file(path_to_data + "/maps/Texas_County_Boundaries_Detailed/Texas_County_Boundaries_Detailed.shp")
TX_ = TX_[['CNTY_NM', 
           'geometry']].rename(columns={'CNTY_NM': 'County'}).to_crs(epsg=4326)

df_ = pd.read_excel(path_to_data + "/gridstatus/meta/Wind_Solar_Regions_to_County.xlsx")

# Merge region info
TX_ = TX_.merge(df_, on="County", how="left")

# Dissolve once (this is enough)
TX_ = TX_.dissolve(by="Wind Region", as_index=False)

# Fix topology
TX_["geometry"] = TX_.buffer(0)

# Optional: rename AFTER everything
TX_ = TX_.rename(columns={'Wind Region': 'region'})

In [3]:
# url = "https://services3.arcgis.com/fwwoCWVtaahwlvxO/arcgis/rest/services/ERCOT_Load_Zones/FeatureServer/7/query"

# params = {"where": "1=1",
#           "outFields": "*",
#           "outSR": "4326",
#           "f": "geojson"}

# data = requests.get(url, params=params).json()

# ERCOT_ = gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")

# # clean columns
# ERCOT_ = ERCOT_.drop(columns=['OBJECTID',
#                               'STATEFP',
#                               'STUSPS',
#                               'AFFGEOID',
#                               'STATENS',
#                               'ALAND',
#                               'AWATER',
#                               'LSAD',
#                               'GEOID',
#                               'Shape__Area',
#                               'Shape__Length',
#                               'ZONE_NAME'])

# ERCOT_ = ERCOT_.rename(columns={'NAME': 'hub'})

# # Panhandle extraction
# panhandle_bbox = box(-103.5, 34.0, -100.0, 36.5)

# panhandle = ERCOT_[ERCOT_["hub"] == "West"].intersection(panhandle_bbox)
# panhandle_union = panhandle.unary_union.buffer(0)

# # Remove from West
# west = ERCOT_[ERCOT_["hub"] == "West"].copy()
# west["geometry"] = west.geometry.difference(panhandle_union)

# # clean west
# west = west[~west.is_empty]

# # Create Panhandle
# panhandle_gdf = gpd.GeoDataFrame({"hub": ["Panhandle"]},
#                                  geometry=[panhandle_union],
#                                  crs=ERCOT_.crs)

# # Combine
# gdf_no_west = ERCOT_[ERCOT_["hub"] != "West"]

# ERCOT_ = gpd.pd.concat([gdf_no_west, west, panhandle_gdf],
#                        ignore_index=True)

# # Save
# ERCOT_.to_file(path_to_data + "/maps/ERCOT_load_zones.shp", 
#                driver="ESRI Shapefile")

In [4]:
B_ = pd.read_excel(path_to_data + "/gridstatus/meta/Jan_2026_generators.xlsx", 
                   sheet_name="Operating")

B_ = B_.loc[B_['Balancing Authority Code'] == 'ERCO'].reset_index(drop = True)
print(B_.shape)

B_ = B_[['Plant Name', 
         'Technology',
         'Balancing Authority Code', 
         'Nameplate Capacity (MW)',
         'Latitude', 
         'Longitude', 
         'Operating Month', 
         'Operating Year']]

B_ = B_.loc[B_['Technology'] == 'Onshore Wind Turbine'].reset_index(drop = True)

B_ = B_.sort_values(["Operating Year", 
                     "Operating Month"], ascending=[True, True])

B_ = B_.drop(columns = ['Technology', 
                        'Balancing Authority Code'])

B_ = gpd.GeoDataFrame(B_, 
                      geometry=gpd.points_from_xy(B_["Longitude"], B_["Latitude"]),
                      crs="EPSG:4326")
    
B_ = gpd.sjoin(B_, TX_[["region", "geometry"]],
               how="left",
               predicate="within")

B_ = B_.drop(columns = ['geometry', 
                        'index_right']).reset_index(drop = True)

print(B_.loc[B_["region"].isna()])
print(B_)

Operating
Canceled or Postponed
Operating_PR
Planned
Planned_PR
Retired
Retired_PR
(2165, 37)
Empty DataFrame
Columns: [Plant Name, Nameplate Capacity (MW), Latitude, Longitude, Operating Month, Operating Year, region]
Index: []
                         Plant Name Nameplate Capacity (MW)   Latitude  \
0    Big Spring Wind Power Facility                    34.3  32.207500   
1         NWP Indian Mesa Wind Farm                    82.5  30.931467   
2        King Mountain Wind Ranch 1                     278  31.209200   
3              Woodward Mountain II                      78  30.951400   
4               Woodward Mountain I                      82  30.951400   
..                              ...                     ...        ...   
204         Prairie Switch Wind LLC                   163.2  29.091900   
205                       Hart Wind                   166.4  34.378546   
206                  Lane City Wind                   202.5  29.209729   
207                  Monte Cris

In [5]:
edges_ = [("Panhandle",  "North"),
          ("Panhandle",  "West"),
          ("North",  "Coastal"),
          ("North",  "South"),
          ("North",  "West"),
          ("South", "Coastal"),
          ("South", "West")]

df_edges_ = pd.DataFrame(edges_, columns=["source", "target"])
print(df_edges_)

region_coords_        = TX_.copy()
region_coords_["lon"] = centroids.x.values
region_coords_["lat"] = centroids.y.values
region_coords_        = region_coords_[["region", "lat", "lon"]]
print(region_coords_)

adjacent_regions_ = {"Coastal": [0, 1, 3],
                     "North": [0, 1, 2, 3, 4],
                     "Panhandle": [1, 2, 4],
                     "South": [0, 1, 3, 4],
                     "West": [1, 2, 3, 4]}
print(adjacent_regions_)

      source   target
0  Panhandle    North
1  Panhandle     West
2      North  Coastal
3      North    South
4      North     West
5      South  Coastal
6      South     West


NameError: name 'centroids' is not defined

In [ ]:
import networkx as nx

fig, ax = plt.subplots(figsize=(3.75, 3.75))

G = nx.from_pandas_edgelist(df_edges_, "source", "target")

# plot regions
regions_ = TX_['region'].unique()
for region, i in zip(regions_, range(len(regions_))):
    TX_.iloc[[i]].plot(ax=ax,
                          fc=palette_['ibm'].iloc[i],
                          ec='None',
                          lw=0,
                          alpha=0.75)

ax.scatter(B_["Longitude"], B_["Latitude"],
           c='k',
           marker = 'o',
           s = .25)
    
for u, v, d in G.edges(data=True):
    x_ = region_coords_.loc[region_coords_['region'] == u, ['lon', 'lat']].to_numpy()[0]
    y_ = region_coords_.loc[region_coords_['region'] == v, ['lon', 'lat']].to_numpy()[0]

    ax.plot([x_[0], y_[0]], [x_[1], y_[1]],
            lw=1.,
            c='gray',
            #marker = 'o',
            #ms = 5.,
            alpha=.5)
    
for region in region_coords_['region'].unique():
    x_ = region_coords_.loc[region_coords_['region'] == region, ['lon', 'lat']].to_numpy()[0]
    
    ax.text(x_[0], x_[1], region,
            ha="center",
            va="center",
            fontsize=12,
            weight="bold")

ax.set_axis_off()

plt.show()

In [ ]:
B_lz_ = B_.groupby(['region', 
                    'Operating Month', 
                    'Operating Year']).agg({'Nameplate Capacity (MW)': 'sum'}).reset_index(drop = False)

B_lz_["date"] = pd.to_datetime(dict(year=B_lz_["Operating Year"].astype(int),
                                    month=B_lz_["Operating Month"].astype(int),
                                    day=1))

B_lz_ = B_lz_.drop(columns = ["Operating Year", 
                              "Operating Month"])

B_lz_ = B_lz_.sort_values(["region", 
                           "date"]).reset_index(drop = True)

B_lz_["Nameplate Capacity (MW)"] = pd.to_numeric(B_lz_["Nameplate Capacity (MW)"])
B_lz_ = B_lz_.rename(columns = {'Nameplate Capacity (MW)': 'capacity_mw'})

# full monthly range across your dataset
full_dates = pd.date_range(start=B_lz_["date"].min(),
                           end=B_lz_["date"].max(),
                           freq="MS")   # Month Start

# 2. full grid: region × date
idx = pd.MultiIndex.from_product([B_lz_["region"].unique(), full_dates],
                                 names=["region", "date"])

# 3. reindex properly
B_lz_full_ = B_lz_.set_index(["region", "date"]).reindex(idx).reset_index()

# 4. fill missing capacity
B_lz_full_["capacity_mw"] = B_lz_full_["capacity_mw"].fillna(0)

# 5. cumulative sum
B_lz_full_ = B_lz_full_.sort_values(["region", "date"])

B_lz_full_["cum_capacity_mw"] = B_lz_full_.groupby("region")["capacity_mw"].cumsum()

# Extra. cumulative sum
B_lz_ = B_lz_.sort_values(["region", "date"])
B_lz_["cum_capacity_mw"] = B_lz_.groupby("region")["capacity_mw"].cumsum()

B_lz_full_ = B_lz_full_.merge(region_coords_,
                              on="region",
                              how="left")

B_lz_full_.to_csv(path_to_data + "/gridstatus/meta/ERCOT_region.csv", 
                  index=False)

In [ ]:
plt.figure(figsize = (3.75, 3.75))

for region, i in zip(regions_, range(len(regions_))):
        
    plt.plot(B_lz_full_.loc[B_lz_full_['region'] == region, 'date'],
             B_lz_full_.loc[B_lz_full_['region'] == region, 'cum_capacity_mw']/1000.,
             label=region,
             color=palette_['ibm'].iloc[i], 
             lw=2)

plt.legend(title="ERCOT (Regions)", 
           frameon = False, 
           title_fontsize=12, 
           fontsize = 10)

plt.xlabel("Date")
plt.ylabel("Cumulative Capacity (GW)")
plt.xlim(B_lz_full_['date'].min(), B_lz_full_['date'].max())
plt.ylim(0, )
plt.tight_layout()

plt.show()

In [ ]:
# WGRPP: Wind Generation Resource Production Potential
# STWPF: Short-Term Wind Power Forecast
# COP_HSL: Current Operating Plan High Sustainable Limit

def _extract_date_from_name(file):
    name = os.path.basename(file).replace(".csv", "")
    name = os.path.basename(name).replace("fc_", "")
    return datetime.strptime(name, "%Y-%m")  


B_lz_ = pd.read_csv(path_to_data + "/gridstatus/meta/ERCOT_region.csv")

B_lz_["region"] = B_lz_["region"].str.lower()
B_lz_["date"] = pd.to_datetime(B_lz_["date"]).dt.strftime("%Y-%m")
regions_ = ['coastal', 
            'north',
            'panhandle',
            'south',
            'west']

fc_type = 'stwpf'
print(regions_)
fc_regions_ = [f'{fc_type}_{region}' for region in regions_]

# Get all CSV files
fc_files_ = glob.glob(os.path.join(path_to_data + "/gridstatus", "fc*.csv"))

# Sort files by date
fc_files_sorted_ = sorted(fc_files_, key=_extract_date_from_name)

print(fc_regions_)

DT_fc_ = []
FC_ = []
for file in fc_files_sorted_:
    print(file)

    df_ = pd.read_csv(file)

    for region, fc_region in zip(regions_, fc_regions_):

        date = file.split('/')[-1].replace(".csv", "").replace("fc_", "")
        idx_ = (B_lz_["region"] == region) & (B_lz_["date"] == date)

        capacity_mw = B_lz_.loc[idx_, "cum_capacity_mw"].to_numpy()[0]
        print(date, region, capacity_mw)

        df_[fc_region] = df_[fc_region].to_numpy()/capacity_mw

        dt_ = df_[['interval_start_utc', 
                   'publish_time_utc']].copy()
        
        fc_ = df_[fc_regions_].copy()
    
        remove_publish_time_local_ = []
        for publish_time_local in dt_['publish_time_utc'].unique():
            idx_ = dt_['publish_time_utc'] == publish_time_local
            if idx_.sum() != 48:
                remove_publish_time_local_.append(publish_time_local)
        
        idx_ = dt_["publish_time_utc"].isin(remove_publish_time_local_)

        dt_ = dt_[~idx_].reset_index(drop=True)
        fc_ = fc_[~idx_].reset_index(drop=True)
        print(fc_.shape, dt_.shape)


    for publish_time_local in dt_['publish_time_utc'].unique():
        idx_ = dt_['publish_time_utc'] == publish_time_local
    
        DT_fc_.append(dt_.loc[idx_, 'interval_start_utc'].to_numpy())
        FC_.append(fc_.loc[idx_, fc_regions_].to_numpy())

DT_fc_ = np.stack(DT_fc_)
FC_ = np.stack(FC_)
print(DT_fc_.shape, FC_.shape)

In [ ]:
# Put into DataFrame
df_ = pd.DataFrame({"time": pd.to_datetime(DT_fc_[:, 0])})

# Extract date (UTC-safe)
df_["date"] = df_["time"].dt.floor("D")

# Count hours per day
counts = df_.groupby("date").size()

# Keep only full days (24 hours)
valid_days = counts[counts == 24].index
idx_ = df_["date"].isin(valid_days)

# Filter
DT_fc_clean_ = DT_fc_[idx_, :].reshape(-1, 24, 48)
FC_clean_ = FC_[idx_, :].reshape(-1, 24, 48, 5)

print(DT_fc_clean_.shape, FC_clean_.shape)

In [ ]:

k = 1
for j in range(24):

    fig, ax = plt.subplots(1, 5, figsize=(15, 2), sharey=True)
    print(DT_clean_[k, j, 0])
    for i in range(5):
        ax[i].plot(FC_clean_[k, j, :, i].T)
        ax[i].set_title(B_lz_["region"].unique()[i])
    
    plt.tight_layout()
    plt.show()

In [ ]:
fc_type = 'gen'
fc_regions_ = [f'{fc_type}_{region}' for region in regions_]
print(regions_)
ac_ = pd.read_csv(path_to_data + "/gridstatus/ac_2023-01_2025-12.csv", index_col = 0)

ac_ = ac_[['interval_start_utc'] + fc_regions_]
ac_["interval_start_utc"] = pd.to_datetime(ac_["interval_start_utc"])
ac_ = ac_.loc[ac_["interval_start_utc"].dt.year < 2026]
dt_ = ac_["interval_start_utc"].to_numpy()

dt_prime_ = ac_["interval_start_utc"].dt.strftime("%Y-%m")

for region in regions_:
    print(region)

    for date in np.unique(dt_prime_):
        idx_ = (B_lz_["region"] == region) & (B_lz_["date"] == date)
        capacity_mw = B_lz_.loc[idx_, "cum_capacity_mw"].to_numpy()[0]
        idx_ = dt_prime_ == date
        ac_.loc[idx_, f'{fc_type}_{region}'] = ac_.loc[idx_, f'{fc_type}_{region}']/capacity_mw
    
ac_ = ac_[fc_regions_].to_numpy()
print(ac_.shape, dt_.shape)

In [ ]:
lag = 24
lead = 48
AC_ = []
DT_ac_ = []
for t in range(dt_.shape[0]):
    x_ = dt_[t - lag:t + lead]
    y_ = ac_[t - lag:t + lead, :]

    if x_.shape[0] == (lag + lead):
        AC_.append(y_)
        DT_ac_.append(x_)

DT_ac_ = np.stack(DT_ac_)
AC_ = np.stack(AC_)
print(DT_ac_.shape, AC_.shape)
print(DT_ac_[0, 0], DT_ac_[-1, -1])

# Put into DataFrame
df_ = pd.DataFrame({"time": pd.to_datetime(DT_ac_[:, 0])})

# Extract date (UTC-safe)
df_["date"] = df_["time"].dt.floor("D")

# Count hours per day
counts = df_.groupby("date").size()

# Keep only full days (24 hours)
valid_days = counts[counts == 24].index
idx_ = df_["date"].isin(valid_days)

# Filter
DT_ac_clean_ = DT_ac_[idx_, :].reshape(-1, 24, 72)
AC_clean_ = AC_[idx_, :].reshape(-1, 24, 72, 5)
print(DT_ac_clean_.shape, AC_clean_.shape)

In [ ]:
DT_ac_clean_prime_ = DT_ac_clean_[..., -48:]
print(DT_ac_clean_prime_.shape, DT_fc_clean_.shape)

idx_ac_ = []
idx_fc_ = []
for i in range(DT_fc_clean_.shape[0]):
    dt_fc_ = pd.to_datetime(DT_fc_clean_[i, 0, -1]).to_numpy()

    for j in range(DT_ac_clean_prime_.shape[0]):
        dt_ac_ = pd.to_datetime(DT_ac_clean_prime_[j, 0, -1]).to_numpy()

        if np.isin(dt_fc_, dt_ac_).all():
            idx_fc_.append(i)
            idx_ac_.append(j)
            break

idx_ac_ = np.array(idx_ac_)
idx_fc_ = np.array(idx_fc_)

print(idx_ac_.shape, idx_fc_.shape)

In [191]:
_DATA = {}

for region in adjacent_regions_.keys():
    idx_ = adjacent_regions_[region]
    print(region, idx_)

    DT_AC_ = np.concatenate([DT_ac_clean_[idx_ac_, ...] for i in idx_], axis = 0)
    DT_FC_ = np.concatenate([DT_fc_clean_[idx_fc_, ...] for i in idx_], axis = 0)
    print(DT_AC_.shape, DT_FC_.shape)
    
    AC_ = np.concatenate([AC_clean_[idx_ac_, ..., i] for i in idx_], axis = 0)
    FC_ = np.concatenate([FC_clean_[idx_fc_,..., i] for i in idx_], axis = 0)
    print(AC_.shape, FC_.shape)

    

# _DATA[region] = {'datetime_actuals': DT_AC_,
#                  'actuals': AC_, 
#                  'datetime_forecasts': DT_FC_,
#                  'forecasts': FC_}
# print(_DATA[region].keys())

# # save
# with open(path_to_data + "/gridstatus/processed_ERCOT_wind_data.pkl", "wb") as f:
#     pickle.dump(_DATA, f)

Coastal [0, 1, 3]
(405, 24, 72) (405, 24, 48)
(3, 405, 24, 72) (3, 405, 24, 48)
dict_keys(['datetime_actuals', 'actuals', 'datetime_forecasts', 'forecasts'])
North [0, 1, 2, 3, 4]
(405, 24, 72) (405, 24, 48)
(5, 405, 24, 72) (5, 405, 24, 48)
dict_keys(['datetime_actuals', 'actuals', 'datetime_forecasts', 'forecasts'])
Panhandle [1, 2, 4]
(405, 24, 72) (405, 24, 48)
(3, 405, 24, 72) (3, 405, 24, 48)
dict_keys(['datetime_actuals', 'actuals', 'datetime_forecasts', 'forecasts'])
South [0, 1, 3, 4]
(405, 24, 72) (405, 24, 48)
(4, 405, 24, 72) (4, 405, 24, 48)
dict_keys(['datetime_actuals', 'actuals', 'datetime_forecasts', 'forecasts'])
West [1, 2, 3, 4]
(405, 24, 72) (405, 24, 48)
(4, 405, 24, 72) (4, 405, 24, 48)
dict_keys(['datetime_actuals', 'actuals', 'datetime_forecasts', 'forecasts'])
